# DriveMind AI — Data Analysis

Exploratory analysis of the synthetic automotive NLU dataset generated by `scripts/generate_dataset.py`.
All numbers in this notebook come directly from the actual dataset on disk — nothing is hard-coded.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.evaluation.model_evaluation import load_annotated_dataframe
from src.config.settings import load_yaml_config

cfg = load_yaml_config('config.yaml')['dataset']
df = load_annotated_dataframe(Path.cwd().parent / cfg['annotated_path'])
df.head()

## Dataset size and split breakdown

In [ ]:
print(f'Total examples: {len(df)}')
print(df['split'].value_counts())
print(f"Unique intents: {df['intent'].nunique()}")

## Intent class distribution (imbalance is real, not synthetic-balanced)

In [ ]:
counts = df['intent'].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 9))
counts.plot(kind='barh', ax=ax, color='#3B6EA5')
ax.set_title('Intent class distribution')
ax.set_xlabel('count')
plt.tight_layout()
plt.show()
print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}")

## Text length distribution

In [ ]:
lengths = df['text'].str.len()
fig, ax = plt.subplots(figsize=(6, 4))
lengths.hist(bins=30, ax=ax, color='#4C9F70')
ax.set_title('Utterance length distribution (characters)')
plt.show()
print(lengths.describe())

## Difficulty breakdown per intent

In [ ]:
pd.crosstab(df['intent'], df['difficulty'])

## Sample inspection: a few random examples per split

In [ ]:
for split in ['train', 'val', 'test']:
    print(f'--- {split} ---')
    sample = df[df['split'] == split].sample(3, random_state=1)
    for _, row in sample.iterrows():
        print(f"  [{row['intent']}] {row['text']!r} entities={row['entities']}")

## Data quality report (computed via src.annotation.quality)

In [ ]:
from src.annotation.quality import analyze_dataset
from src.annotation.validator import validate_dataset

report = analyze_dataset(df)
print(f'Total samples: {report.total_samples}')
print(f'Duplicate samples: {report.duplicate_samples}')
print(f'Missing labels: {report.missing_labels}')
print(f'Class imbalance ratio: {report.class_imbalance_ratio}')
print(f'Avg text length: {report.avg_text_length}')
print(f'Entity coverage: {report.entity_coverage}')

In [ ]:
records = df.to_dict(orient='records')
validation_report = validate_dataset(records)
print(f'Valid: {validation_report.valid} / {validation_report.total}')
print('Issue breakdown:', dict(validation_report.issue_counts()))